# 🔬 Notebook 3: Flash Sale — Deep Dive: Atomic stock, rate limit, admission queue

## 🛠️ Setup

```bash
cd 06-system-designs/flash-sale
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Deep dive 1 — atomic stock with a Redis Lua-like script

The core trick is that **decrement and check** must be atomic. Without atomicity, two requests
can both read `stock=1`, both decrement, and we oversell.

Redis has `DECR` which is atomic by itself; combined with `GET` it usually still is,
but the safest pattern is **Lua**:

```lua
-- KEYS[1] = stock:item-42
local remaining = tonumber(redis.call('GET', KEYS[1]))
if remaining > 0 then
  redis.call('DECR', KEYS[1]); return 1
else
  return 0
end
```

We'll simulate this in Python with a lock.

In [ ]:
import threading, random, time

class AtomicStock:
    def __init__(self, initial): self.n = initial; self.lock = threading.Lock()
    def try_reserve(self):
        with self.lock:
            if self.n > 0:
                self.n -= 1
                return True
            return False

stock = AtomicStock(1000)
reserved = [0]
attempts = [0]
def buyer():
    for _ in range(50):
        attempts[0] += 1
        if stock.try_reserve(): reserved[0] += 1
        # simulate slight jitter
        time.sleep(random.uniform(0, 0.0005))

threads = [threading.Thread(target=buyer) for _ in range(100)]  # 100 threads × 50 = 5000 attempts
t0 = time.time()
for t in threads: t.start()
for t in threads: t.join()
print(f"Attempts: {attempts[0]}, Reserved: {reserved[0]}, Stock left: {stock.n} "
      f"({time.time()-t0:.2f}s)")
assert reserved[0] == 1000 and stock.n == 0, "no overselling"
print("✓ exactly 1000 reserved, no overselling")


## Deep dive 2 — token-bucket rate limiter per user

In [ ]:
import time

class TokenBucket:
    def __init__(self, capacity: int, refill_per_s: float):
        self.capacity = capacity
        self.tokens = float(capacity)
        self.rate = refill_per_s
        self.last = time.time()

    def allow(self, n=1) -> bool:
        now = time.time()
        self.tokens = min(self.capacity, self.tokens + (now - self.last) * self.rate)
        self.last = now
        if self.tokens >= n:
            self.tokens -= n; return True
        return False

# 5 reqs per second, burst 3
tb = TokenBucket(capacity=3, refill_per_s=5)
results = []
for i in range(10):
    ok = tb.allow(); results.append(ok); time.sleep(0.1)
print("allowed:", results)
print("→ first 3 allowed (burst), then ~1 per 200ms (refill rate)")


## Deep dive 3 — why a queue in front of stock?

Without a queue, millions of threads compete on the same Redis key. Even with Lua, you
saturate the single-threaded Redis instance.

**Bounded admission queue** solves two problems:
1. **Backpressure**: if queue is full, return 503/out-of-stock immediately — no server melts.
2. **Batch commits**: a consumer can process messages in small batches and use pipelining.

Queue size ≈ stock × 1.5. Once it's full, the sale is effectively over for newcomers.

### Fairness note
"First come first served" at planetary scale is actually a *user experience* illusion — the
internet doesn't provide global ordering. We settle for: if you arrived before the queue filled,
you get a chance; otherwise, sorry.
